# 03 YOLO Training, Evaluation, and Export

This is the main YOLO26 workflow. It trains the RGB segmentation model, optionally trains the RGB+HSV variant when that dataset exists, evaluates available models, and exports the selected model to NCNN for Raspberry Pi deployment.

Run `01_synthetic_generation.ipynb` first. Run `02_colour_space_rgb_hsv_experiment.ipynb` first as well if you want the RGB+HSV comparison.


## Setup

Training settings such as model name, epochs, image size, batch size, optimizer, patience, and augmentation values come from `config/project.yaml`.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

from naartjie_cv.config import class_names, load_config, project_root, resolve_path

ROOT = project_root()
CONFIG = ROOT / 'config' / 'project.yaml'
cfg = load_config(CONFIG)

print(f'Project root: {ROOT}')
print(f'Config: {CONFIG}')


## Runtime Check

This verifies the active conda environment, PyTorch, CUDA, GPU, and Ultralytics installation before training.


In [ ]:
from naartjie_cv.runtime import assert_training_runtime, print_runtime_report

report = assert_training_runtime()
print_runtime_report(report)


## Dataset Check

The RGB dataset is required. The RGB+HSV dataset is optional here and is used only when it exists.


In [ ]:
rgb_data_yaml = resolve_path(cfg, cfg['paths']['synthetic_dir']) / 'data.yaml'
rgb_hsv_data_yaml = resolve_path(cfg, cfg['paths']['synthetic_multispectral_dir']) / 'data.yaml'

if not rgb_data_yaml.exists():
    raise FileNotFoundError(f'RGB synthetic dataset not found: {rgb_data_yaml}. Run 01_synthetic_generation.ipynb first.')

print(f'RGB dataset: {rgb_data_yaml}')
print(f'RGB+HSV dataset: {rgb_hsv_data_yaml if rgb_hsv_data_yaml.exists() else "not found"}')


## Train RGB YOLO26 Instance Segmentation

If `runs/segment/naartjie_rgb/weights/last.pt` exists and `resume: true` is enabled in the config, this resumes the interrupted run automatically.


In [ ]:
from naartjie_cv.train import train_yolo

rgb_results = train_yolo(CONFIG, variant='rgb')
rgb_results


## Optional: Train RGB+HSV YOLO26

Set `TRAIN_RGB_HSV = True` if you want this notebook to train the RGB+HSV variant too. Leave it `False` if you already handled that in `02_colour_space_rgb_hsv_experiment.ipynb`.


In [ ]:
TRAIN_RGB_HSV = False

if TRAIN_RGB_HSV and rgb_hsv_data_yaml.exists():
    rgb_hsv_results = train_yolo(CONFIG, variant='rgb_hsv')
elif TRAIN_RGB_HSV:
    rgb_hsv_results = None
    print('Skipping RGB+HSV training because the RGB+HSV dataset does not exist.')
else:
    rgb_hsv_results = None
    print('RGB+HSV training is disabled in this notebook. Set TRAIN_RGB_HSV = True to run it here.')

rgb_hsv_results


## Evaluate Available YOLO Models

Evaluation saves JSON and CSV reports under `runs/evaluations`.


In [ ]:
from naartjie_cv.evaluate import evaluate_model

rgb_weights = ROOT / 'runs' / 'segment' / 'naartjie_rgb' / 'weights' / 'best.pt'
rgb_hsv_weights = ROOT / 'runs' / 'segment' / 'naartjie_rgb_hsv' / 'weights' / 'best.pt'

reports = {}
if rgb_weights.exists():
    reports['rgb_test'] = evaluate_model(CONFIG, rgb_weights, variant='rgb', split='test')
else:
    print(f'Skipping RGB evaluation; weights not found: {rgb_weights}')

if rgb_hsv_weights.exists():
    reports['rgb_hsv_test'] = evaluate_model(CONFIG, rgb_hsv_weights, variant='rgb_hsv', split='test')
else:
    print(f'Skipping RGB+HSV evaluation; weights not found: {rgb_hsv_weights}')

print(json.dumps(reports, indent=2))


## Export Selected Model to NCNN

Export the model selected after comparing RGB and RGB+HSV metrics. The RGB model is the reliable Raspberry Pi fallback.


In [ ]:
from naartjie_cv.export import export_ncnn

SELECTED_VARIANT = 'rgb'  # change to 'rgb_hsv' only after validation passes
SELECTED_WEIGHTS = rgb_weights if SELECTED_VARIANT == 'rgb' else rgb_hsv_weights

if not SELECTED_WEIGHTS.exists():
    raise FileNotFoundError(f'Selected weights not found: {SELECTED_WEIGHTS}')

exported = export_ncnn(CONFIG, SELECTED_WEIGHTS, variant=SELECTED_VARIANT)
exported


## Raspberry Pi Command

After copying the exported INT8 NCNN model directory and `background/ncnn_infer` app to the Raspberry Pi, run:

```bash
./build/ncnn_infer --model /path/to/best_ncnn_model --input /path/to/images --output /path/to/results --conf 0.25 --imgsz 640 --source-width 640 --source-height 480 --threads 4 --classes class_1,class_2,class_3
```

Use `--channels rgbhsv` only for a validated RGB+HSV NCNN export.
